# 001 Hybrid Retrieval

这个 notebook 用密云项目的真实 ES 与 embedding/reranker 服务配置，轻量还原混合检索。

注意：这里不直接 import 密云项目的 `MyElasticsearchStore`，因为 `fastapi-study` 和密云项目的 LangChain 版本不同。教学目标是理解流程，所以用少量 Python 代码复刻核心逻辑：

```text
BM25 检索 + 语义向量检索 -> RRF 融合 -> reranker 重排 -> 返回 topN
```

## 0. 对照密云项目源码

密云项目里的真实调用链是：

```text
local_search.py::local_search
  -> update_config(requests)
  -> get_qa_bot_with_docs_chain(requests)
  -> itemgetter("question") | retriever
  -> vectorstores.py::ES_ParentDocumentRetriever._aget_relevant_documents
  -> utils.py::bm25_retrieve / semantic_retrieval / multi_way_retrieve
```

本 notebook 不复用这些类，只复刻关键算法和 ES 查询方式，避免版本差异影响学习。

## 1. 读取密云项目真实运行配置

In [48]:
from pathlib import Path
import importlib
import sys

module_dir_candidates = [
    Path.cwd(),
    Path.cwd() / "notebooks" / "es_way_retrieve",
    Path("/home/dev/bxc/fastapi-study/notebooks/es_way_retrieve"),
]

for module_dir in module_dir_candidates:
    if (module_dir / "project_env.py").exists():
        sys.path.insert(0, str(module_dir))
        break
else:
    raise FileNotFoundError("找不到 project_env.py")

import project_env
project_env = importlib.reload(project_env)
from project_env import load_miyun_runtime_config, create_elasticsearch_client

runtime = load_miyun_runtime_config()
runtime

MiyunRuntimeConfig(project_root=PosixPath('/home/dev/bxc/miyun_pro/miyun_pro'), es_addresses='http://192.168.102.19:9200', es_user='elastic', embedding_base_url='http://192.168.102.19:8082/v1', analyzer='ik_smart')

In [34]:
es = create_elasticsearch_client(runtime)
info = es.info()
print("ES:", runtime.es_addresses)
print("cluster:", info.get("cluster_name"), "version:", info.get("version", {}).get("number"))
print("embedding/reranker:", runtime.embedding_base_url)
print("analyzer:", runtime.analyzer)

ES: http://192.168.102.19:9200
cluster: docker-cluster version: 8.14.2
embedding/reranker: http://192.168.102.19:8082/v1
analyzer: ik_smart


## 2. 为什么 BM25 需要分词器

BM25 看的是词项命中。中文如果不分词，`密云水库泄洪通知` 很可能被当成一个整体，关键词检索效果会很差。

密云项目创建索引时使用 `ik_smart`。

In [49]:
sample_query = "密云水库泄洪通知"

tokens = es.indices.analyze(
    body={"text": sample_query, "analyzer": runtime.analyzer}
)["tokens"]

[token["token"] for token in tokens]

['密云', '水库', '泄洪', '通知']

## 3. 准备查询参数

改这里就可以对不同知识库做实验。`ES_INDEX`、`ES_QUERY`、`TENANT_ID`、`FILE_ID` 都可以用环境变量覆盖。

In [50]:
import os

index_name = os.getenv("ES_INDEX", "1833773738085195776_prod")
#query = os.getenv("ES_QUERY", sample_query)
query = os.getenv("ES_QUERY", "防汛抢险新技术的应用与研究有哪些内容")
tenant_id = os.getenv("TENANT_ID", "")
file_id = os.getenv("FILE_ID", "")
#embedding_model = os.getenv("EMBEDDING_MODEL", "Conan-embedding-v1")
embedding_model = "Conan-embedding-v1"
reranker_model = os.getenv("RERANKER_MODEL", "bge-reranker-base")

filters = []
if tenant_id and tenant_id != "string":
    filters.append({"match": {"tenant_id": tenant_id}})
if file_id:
    filters.append({"terms": {"file_id.keyword": file_id.split(",")}})

print("index_name:", index_name)
print("query:", query)
print("filters:", filters)
print("embedding_model:", embedding_model)
print("reranker_model:", reranker_model)

index_name: 1833773738085195776_prod
query: 防汛抢险新技术的应用与研究有哪些内容
filters: []
embedding_model: Conan-embedding-v1
reranker_model: bge-reranker-base


## 4. BM25 检索

密云项目对应：`CRUDMixin.sync_bm25_retrieve()` / `bm25_retrieve()`。

In [51]:
def bm25_search(es, index_name, query, filters=None, size=10):
    filters = list(filters or [])
    body = {
        "query": {
            "bool": {
                "must": [{"match": {"text": {"query": query}}}] + filters,
                "must_not": [{"term": {"state": False}}],
            }
        },
        "highlight": {
            "fields": {
                "text": {
                    "pre_tags": ["<strong>"],
                    "post_tags": ["</strong>"],
                    "fragment_size": 200,
                }
            }
        },
    }
    response = es.search(index=index_name, body=body, size=size)
    docs = []
    for hit in response["hits"]["hits"]:
        source = hit.get("_source", {})
        metadata = source.get("metadata", {}) or {}
        metadata.update({
            "_id": hit.get("_id"),
            "_index": hit.get("_index"),
            "score": hit.get("_score"),
            "search_type": "BM25",
            "highlight": "".join(hit.get("highlight", {}).get("text", [])) or None,
        })
        docs.append({"page_content": source.get("text", ""), "metadata": metadata})
    return docs

bm25_docs = bm25_search(es, index_name, query, filters=filters, size=10)
for i, doc in enumerate(bm25_docs[:5], 1):
    print("=" * 80)
    print("BM25", i, "score=", doc["metadata"].get("score"), "file=", doc["metadata"].get("file_name"))
    print(doc["metadata"].get("highlight") or doc["page_content"][:300])

BM25 1 score= 18.642815 file= 防汛抢险技术.pdf
200 <strong>防</strong><strong>汛</strong><strong>抢</strong><strong>险</strong><strong>技</strong><strong>术</strong><strong>险</strong>队自身负责维护和更<strong>新</strong>，走自我发展<strong>的</strong>良性循环之路。(三)人员<strong>的</strong>组织和管理对以地(市)级河务局为单位组建<strong>的</strong><strong>抢</strong><strong>险</strong>队，人员<strong>应</strong>在全局范围<strong>内</strong>抽调，使来源更广泛<strong>些</strong>。人员配备按照分工合作、一专多能<strong>的</strong>原则，年龄为18岁~45岁，结构比例合适 队员<strong>应</strong>具备初中以上文化，<strong>有</strong>一定<strong>的</strong><strong>防</strong><strong>汛</strong><strong>抢</strong><strong>险</strong>经验，对一<strong>些</strong>重要岗位，人员要达到中专或大专文化程度。对<strong>抢</strong><strong>险</strong>队员<strong>的</strong>培训<strong>应</strong><strong>有</strong>保证，培训<strong>内</strong><strong>容</strong>需要增加<strong>抢</strong><strong>险</strong>机械设备维修和管理，使每一个队员都能够达到懂机械、会维修<strong>的</strong>要求。(四)提高和改善<strong>抢</strong><strong>险</strong>队<strong>的</strong>生产和生活条件黄河<strong>抢</strong><strong>险</strong>是一项很艰苦<strong>的</strong>工作，关键时刻还

/tmp/ipykernel_4086317/207095270.py:20: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  response = es.search(index=index_name, body=body, size=size)


## 5. 语义向量检索

密云项目对应：`CRUDMixin.semantic_retrieval()`。这里直接用 OpenAI-compatible embedding 服务拿 query vector，再用 ES `knn` 查 `vector` 字段。

In [52]:
from openai import OpenAI

embedding_client = OpenAI(
    api_key=os.getenv("EMBEDDING_API_KEY", "EMPTY"),
    base_url=runtime.embedding_base_url,
)

def embed_query(text):
    response = embedding_client.embeddings.create(model=embedding_model, input=[text])
    return response.data[0].embedding

def semantic_search(es, index_name, query, filters=None, k=10, num_candidates=100):
    query_vector = embed_query(query)
    knn = {
        "field": "vector",
        "query_vector": query_vector,
        "k": k,
        "num_candidates": num_candidates,
    }
    if filters:
        knn["filter"] = filters
    body = {"knn": knn, "size": k}
    response = es.search(index=index_name, body=body)
    print(f"Semantic search response: {response}")
    docs = []
    for hit in response["hits"]["hits"]:
        source = hit.get("_source", {})
        metadata = source.get("metadata", {}) or {}
        metadata.update({
            "_id": hit.get("_id"),
            "_index": hit.get("_index"),
            "score": hit.get("_score"),
            "search_type": "SEMANTIC",
        })
        docs.append({"page_content": source.get("text", ""), "metadata": metadata})
    return docs

semantic_docs = semantic_search(es, index_name, query, filters=filters, k=10)
for i, doc in enumerate(semantic_docs[:5], 1):
    print("=" * 80)
    print("SEMANTIC", i, "score=", doc["metadata"].get("score"), "file=", doc["metadata"].get("file_name"))
    print(doc["page_content"][:300])

Semantic search response: {'took': 8, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 10, 'relation': 'eq'}, 'max_score': 0.93739843, 'hits': [{'_index': '1833773738085195776_prod', '_id': 'a1b2c797-44b6-47ea-96ff-32057da532ed', '_score': 0.93739843, '_ignored': ['parent_text.keyword', 'metadata.parent_text.keyword', 'text.keyword'], '_source': {'text': '642 防汛抢险技术探测隐患技术和组合压力灌浆技术又取代了锥探灌浆，从而大幅度提高了堤防隐患探测和加固的效率和质量。进入 80年代以后，防汛抢险的新技术、新结构和新材料研究成为人们关注的热点。各种形式的沉排坝(包括化纤编织袋沉排坝、长管袋充填泥浆沉排坝、铅丝笼沉排坝、网护根坝、褥垫式沉排坝、柳石枕沉排坝、铰链式混凝土模袋沉排坝、挤压块沉排坝、粘结大块石坝、潜坝等)、混凝土透水桩坝等新坝型在坝岸工程中得到应用。研制了诸如捆枕器、打桩机、机编铅丝网等抢险机具和根石探测、堵漏洞等抢险新技术。这些成果的应用，取得了明显的效果。总之，防洪抢险新技术的应用与研制的主要任务是：为汛情预报、防洪工程建设、防汛部署、紧急抢险、各种险情抢护提供安全、高效的措施方案；对指挥抗洪斗争的关键问题如开闸还是关闸、破堤还是保堤等提出对策；建立高科技立体探测网，如防汛计算机网络、气象卫星雷达系统、航空航天遥感系统等，准确把握洪水的脉搏；通过对水情、雨情及各主要防洪工程设施可靠性的掌握和对天气情况的分析提出迎战洪水预案，为防洪决策提供依据；提高防汛抢险的安全度和减少危险性，提高工作效能和资料的精确度，提高防汛抢险人员的素质和工作能力，增加抢险成功的把握。第二节 防汛新技术的应用与研制一、水文气象情报预报———防汛的耳目(一)黄河暴雨洪水情报预报系统的研究与开发黄河

## 6. RRF 融合

RRF 不直接比较 BM25 分数和向量分数，因为两种分数尺度不同。它只看各自排序名次，再按名次融合。密云项目权重是 `[0.3, 0.9]`，语义检索更高。

In [53]:
def doc_key(doc):
    metadata = doc.get("metadata", {})
    return metadata.get("_id") or f"{metadata.get('file_id')}:{metadata.get('segment_id')}:{doc.get('page_content', '')[:80]}"

def rrf_fuse(doc_lists, weights=None, rank_constant=60):
    weights = weights or [1 / len(doc_lists)] * len(doc_lists)
    scores = {}
    docs_by_key = {}
    sources_by_key = {}
    for list_index, docs in enumerate(doc_lists):
        weight = weights[list_index] if list_index < len(weights) else 1
        for rank, doc in enumerate(docs, start=1):
            key = doc_key(doc)
            docs_by_key.setdefault(key, doc)
            scores[key] = scores.get(key, 0.0) + weight / (rank_constant + rank)
            sources_by_key.setdefault(key, []).append(doc["metadata"].get("search_type"))
    result = []
    for key, score in sorted(scores.items(), key=lambda item: item[1], reverse=True):
        doc = docs_by_key[key]
        doc = {"page_content": doc["page_content"], "metadata": dict(doc["metadata"])}
        doc["metadata"]["rrf_score"] = score
        doc["metadata"]["retrieval_sources"] = sources_by_key[key]
        result.append(doc)
    return result

rrf_docs = rrf_fuse([bm25_docs, semantic_docs], weights=[0.3, 0.9])
for i, doc in enumerate(rrf_docs[:6], 1):
    print("=" * 80)
    print("RRF", i, "rrf_score=", round(doc["metadata"].get("rrf_score"), 6), "sources=", doc["metadata"].get("retrieval_sources"))
    print(doc["page_content"][:220])

RRF 1 rrf_score= 0.018928 sources= ['BM25', 'SEMANTIC']
642 防汛抢险技术探测隐患技术和组合压力灌浆技术又取代了锥探灌浆，从而大幅度提高了堤防隐患探测和加固的效率和质量。进入 80年代以后，防汛抢险的新技术、新结构和新材料研究成为人们关注的热点。各种形式的沉排坝(包括化纤编织袋沉排坝、长管袋充填泥浆沉排坝、铅丝笼沉排坝、网护根坝、褥垫式沉排坝、柳石枕沉排坝、铰链式混凝土模袋沉排坝、挤压块沉排坝、粘结大块石坝、潜坝等)、混凝土透水桩坝等新坝型在坝岸工程中得到应用。研制了诸如捆枕器、打桩机、机
RRF 2 rrf_score= 0.018634 sources= ['BM25', 'SEMANTIC']
642 防汛抢险技术探测隐患技术和组合压力灌浆技术又取代了锥探灌浆，从而大幅度提高了堤防隐患探测和加固的效率和质量。进入 80年代以后，防汛抢险的新技术、新结构和新材料研究成为人们关注的热点。各种形式的沉排坝(包括化纤编织袋沉排坝、长管袋充填泥浆沉排坝、铅丝笼沉排坝、网护根坝、褥垫式沉排坝、柳石枕沉排坝、铰链式混凝土模袋沉排坝、挤压块沉排坝、粘结大块石坝、潜坝等)、混凝土透水桩坝等新坝型在坝岸工程中得到应用。研制了诸如捆枕器、打桩机、机
RRF 3 rrf_score= 0.01854 sources= ['BM25', 'SEMANTIC']
640 防汛抢险技术第十三章 防汛抢险新技术的应用与研究第一节 防汛抢险中新技术的重大作用在抗洪抢险中，科学技术发挥着重要作用。从党中央、国务院的宏观决策到每一个重大行动的部署和实施；从汛情预测到查险排险；从危堤抢险到分蓄洪调度，每一步，每一个环节都离不开科学技术，一大批高新技术成果被应用。成千上万名科技工作者以不同方式投身抗洪，提供了许多出色的技术。开闸还是关闸、保堤还是破堤，这些指挥抗洪斗争的关键问题需要科技智囊团来提出对策。在抗洪
RRF 4 rrf_score= 0.014754 sources= ['SEMANTIC']
642 防汛抢险技术探测隐患技术和组合压力灌浆技术又取代了锥探灌浆，从而大幅度提高了堤防隐患探测和加固的效率和质量。进入 80年代以后，防汛抢险的新技术、新结构和新材料研究成为人们关注的热点。各种形式的沉排坝(包括化纤编织袋沉排坝、长管袋充填泥

## 7. reranker 重排

当前 embedding/reranker 服务虽然是 OpenAI-compatible 服务，但 reranker 暴露的是 `/v1/rerank` 接口，不是 `/v1/embeddings`。另外 `bge-reranker-base` 的单条输入上下文较小，服务端报错显示最大约 `512 tokens`，所以不能把 ES 返回的长段落原样送入 reranker。这里会先截断候选文本，只把用于打分的短文本发给 reranker，返回结果仍保留原始文档内容。

In [55]:
import httpx

def build_rerank_text(doc, max_chars=400):
    metadata = doc.get("metadata", {})
    text = metadata.get("highlight") or doc.get("page_content", "")
    return str(text).replace("<strong>", "").replace("</strong>", "")[:max_chars]

def rerank(query, docs, model=reranker_model, threshold=0.4, limit=8, max_doc_chars=400):
    if not docs:
        return []

    endpoint = f"{str(runtime.embedding_base_url).rstrip('/')}/rerank"
    rerank_texts = [build_rerank_text(doc, max_chars=max_doc_chars) for doc in docs]
    print("rerank docs:", len(rerank_texts), "max_chars:", max(len(text) for text in rerank_texts))
    payload = {
        "model": model,
        "query": query,
        "documents": rerank_texts,
    }
    response = httpx.post(endpoint, json=payload, timeout=60)
    if response.status_code >= 400:
        print("rerank error:", response.text[:800])
    response.raise_for_status()
    data = response.json()

    score_by_index = {
        item["index"]: item["relevance_score"]
        for item in data.get("results", [])
    }
    ranked = []
    for index, doc in enumerate(docs):
        if index not in score_by_index:
            continue
        new_doc = {"page_content": doc["page_content"], "metadata": dict(doc["metadata"])}
        new_doc["metadata"]["rerank_score"] = score_by_index[index]
        ranked.append(new_doc)

    filtered = [doc for doc in ranked if doc["metadata"]["rerank_score"] > threshold]
    if not filtered:
        filtered = ranked
    return sorted(filtered, key=lambda doc: doc["metadata"]["rerank_score"], reverse=True)[:limit]

reranked_docs = rerank(query, rrf_docs[:20])
for i, doc in enumerate(reranked_docs[:6], 1):
    print("=" * 80)
    print("RERANK", i, "score=", round(doc["metadata"].get("rerank_score"), 6), "file=", doc["metadata"].get("file_name"))
    print(doc["page_content"][:260])

rerank docs: 17 max_chars: 400
RERANK 1 score= 0.998873 file= 防汛抢险技术.pdf
640 防汛抢险技术第十三章 防汛抢险新技术的应用与研究第一节 防汛抢险中新技术的重大作用在抗洪抢险中，科学技术发挥着重要作用。从党中央、国务院的宏观决策到每一个重大行动的部署和实施；从汛情预测到查险排险；从危堤抢险到分蓄洪调度，每一步，每一个环节都离不开科学技术，一大批高新技术成果被应用。成千上万名科技工作者以不同方式投身抗洪，提供了许多出色的技术。开闸还是关闸、保堤还是破堤，这些指挥抗洪斗争的关键问题需要科技智囊团来提出对策。在抗洪抢险中，从中央科技部门到各省市都普遍使用了防汛计算机网络、气象卫星雷达系统、航空
RERANK 2 score= 0.998752 file= 防汛抢险技术.pdf
642 防汛抢险技术探测隐患技术和组合压力灌浆技术又取代了锥探灌浆，从而大幅度提高了堤防隐患探测和加固的效率和质量。进入 80年代以后，防汛抢险的新技术、新结构和新材料研究成为人们关注的热点。各种形式的沉排坝(包括化纤编织袋沉排坝、长管袋充填泥浆沉排坝、铅丝笼沉排坝、网护根坝、褥垫式沉排坝、柳石枕沉排坝、铰链式混凝土模袋沉排坝、挤压块沉排坝、粘结大块石坝、潜坝等)、混凝土透水桩坝等新坝型在坝岸工程中得到应用。研制了诸如捆枕器、打桩机、机编铅丝网等抢险机具和根石探测、堵漏洞等抢险新技术。这些成果的应用，取得了明显的效
RERANK 3 score= 0.998752 file= 防汛抢险技术.pdf
642 防汛抢险技术探测隐患技术和组合压力灌浆技术又取代了锥探灌浆，从而大幅度提高了堤防隐患探测和加固的效率和质量。进入 80年代以后，防汛抢险的新技术、新结构和新材料研究成为人们关注的热点。各种形式的沉排坝(包括化纤编织袋沉排坝、长管袋充填泥浆沉排坝、铅丝笼沉排坝、网护根坝、褥垫式沉排坝、柳石枕沉排坝、铰链式混凝土模袋沉排坝、挤压块沉排坝、粘结大块石坝、潜坝等)、混凝土透水桩坝等新坝型在坝岸工程中得到应用。研制了诸如捆枕器、打桩机、机编铅丝网等抢险机具和根石探测、堵漏洞等抢险新技术。这些成果的应用，取得了明显的效
RERANK 4 score= 0.998751 file= 防汛抢险技术.pdf
642 防汛抢险技术探测隐患技术和组

## 8. 封装成混合检索函数

这就是密云项目 `multi_way_retrieve()` 的轻量还原版。

In [56]:
def hybrid_retrieve(query, return_size=6, bm25_size=10, semantic_k=10):
    bm25 = bm25_search(es, index_name, query, filters=filters, size=bm25_size)
    semantic = semantic_search(es, index_name, query, filters=filters, k=semantic_k)
    fused = rrf_fuse([bm25, semantic], weights=[0.3, 0.9])
    reranked = rerank(query, fused[:20])
    return reranked[:return_size]

hybrid_docs = hybrid_retrieve(query)
for i, doc in enumerate(hybrid_docs, 1):
    print("=" * 80)
    print("HYBRID", i, "rerank=", round(doc["metadata"].get("rerank_score", 0), 6), "sources=", doc["metadata"].get("retrieval_sources"))
    print(doc["page_content"][:300])

Semantic search response: {'took': 6, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 10, 'relation': 'eq'}, 'max_score': 0.93739843, 'hits': [{'_index': '1833773738085195776_prod', '_id': 'a1b2c797-44b6-47ea-96ff-32057da532ed', '_score': 0.93739843, '_ignored': ['parent_text.keyword', 'metadata.parent_text.keyword', 'text.keyword'], '_source': {'text': '642 防汛抢险技术探测隐患技术和组合压力灌浆技术又取代了锥探灌浆，从而大幅度提高了堤防隐患探测和加固的效率和质量。进入 80年代以后，防汛抢险的新技术、新结构和新材料研究成为人们关注的热点。各种形式的沉排坝(包括化纤编织袋沉排坝、长管袋充填泥浆沉排坝、铅丝笼沉排坝、网护根坝、褥垫式沉排坝、柳石枕沉排坝、铰链式混凝土模袋沉排坝、挤压块沉排坝、粘结大块石坝、潜坝等)、混凝土透水桩坝等新坝型在坝岸工程中得到应用。研制了诸如捆枕器、打桩机、机编铅丝网等抢险机具和根石探测、堵漏洞等抢险新技术。这些成果的应用，取得了明显的效果。总之，防洪抢险新技术的应用与研制的主要任务是：为汛情预报、防洪工程建设、防汛部署、紧急抢险、各种险情抢护提供安全、高效的措施方案；对指挥抗洪斗争的关键问题如开闸还是关闸、破堤还是保堤等提出对策；建立高科技立体探测网，如防汛计算机网络、气象卫星雷达系统、航空航天遥感系统等，准确把握洪水的脉搏；通过对水情、雨情及各主要防洪工程设施可靠性的掌握和对天气情况的分析提出迎战洪水预案，为防洪决策提供依据；提高防汛抢险的安全度和减少危险性，提高工作效能和资料的精确度，提高防汛抢险人员的素质和工作能力，增加抢险成功的把握。第二节 防汛新技术的应用与研制一、水文气象情报预报———防汛的耳目(一)黄河暴雨洪水情报预报系统的研究与开发黄河

/tmp/ipykernel_4086317/207095270.py:20: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  response = es.search(index=index_name, body=body, size=size)


## 9. 复盘

1. BM25 依赖 `ik_smart` 分词，擅长精确词、编号、专有名词。
2. 语义检索依赖 embedding，擅长同义表达和口语化问题。
3. RRF 只融合排名，不直接比较不同检索方式的原始分数。
4. reranker 是在候选集上做更细的相关性判断，不能替代召回。
5. 生产项目里还会加 tenant/file 过滤、知识图谱召回、引用格式化、SSE 流式输出。